<a href="https://colab.research.google.com/github/JoelForson/ECON5200-Applied-Data-Analytics-in-Economics/blob/main/Lab_12/OLS%2C_Hedonic_Pricing%2C_and_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Environment Initialization and Data Ingestion
Establish your analytical environment by importing all required analytical libraries. Specifically, ensure you import smf for the formula API, which bridges the gap between statistical notation and Python execution. Load the 2026 Zillow dataset directly into a pandas DataFrame and perform a rapid Exploratory Data Analysis (EDA) to understand the scale and variance of the target variable, Home_Value.

In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/RyanPiao/ECON5200-Applied-Data-Analytics-in-Economics/refs/heads/main/Data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

# Step 2: Defining the OLS Architecture via Patsy Formulas
Instead of manually constructing orthogonal matrices, build a multivariate OLS model using the statsmodels.formula.api (smf) module. Write a patsy formula string that regresses Home_Value on Square_Footage, Property_Age, and Distance_to_Transit. The patsy formula automatically handles the inclusion of a constant intercept term.

In [10]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~ Square_Footage+Property_Age+Distance_to_Transit+School_District_Rating'

# Step 3: Model Fitting and Diagnostic Extraction
Fit the model to the data using the .fit() method and print the summary statistics table. Examine the R-squared value and analyze the coefficients. Ensure the signs of the coefficients align with economic intuition (e.g., assessing whether older homes generally depreciate in this market) and check the t-statistics for significance.

In [11]:
# Step 3: Fitting the model and printing the summary
ols1 = smf.ols(formula=formula, data=df)
results = ols1.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Tue, 10 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        18:59:03   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

# Step 4: The Machine Learning Pivot: Generating Predictions
We push past classical explanation and move directly into out-of-sample prediction. Use the model.predict() function to generate a continuous vector of predicted housing values based on your feature matrix.

In [12]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict()

# Step 5: Calculating and Interpreting the Root Mean Squared Error
Import the rmse function directly from statsmodels.tools.eval_measures. Calculate the RMSE between the ground truth and your generated predictions. Print the result formatted cleanly as a US Dollar currency string to interpret the absolute magnitude of the algorithm's financial blind spot.

In [17]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df['Home_Value'],y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [18]:
"""
=============================================================================
  RESIDUAL FORENSICS DASHBOARD — Hedonic Pricing OLS Model
  Powered by statsmodels + Plotly Express
=============================================================================
  PURPOSE:
    Extend a hedonic pricing OLS model with an interactive residual
    diagnostics panel to detect heteroscedasticity, structural breaks,
    and influential outliers.
=============================================================================
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_squared_error

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 ── Synthetic Hedonic Pricing Dataset
#   (Replace this block with your actual dataset / OLS results object)
# ─────────────────────────────────────────────────────────────────────────────
np.random.seed(42)
n = 350

sqft        = np.random.randint(600, 4500, n)
bedrooms    = np.random.randint(1, 6, n)
bathrooms   = np.random.randint(1, 4, n)
age         = np.random.randint(0, 60, n)
dist_cbd    = np.random.uniform(0.5, 30, n)   # miles to central business district
garage      = np.random.randint(0, 2, n)       # binary: 1 = has garage

# True price relationship (log-linear is common in hedonic models)
log_price = (
    12.0
    + 0.0003 * sqft
    + 0.04   * bedrooms
    + 0.06   * bathrooms
    - 0.005  * age
    - 0.03   * dist_cbd
    + 0.08   * garage
    # Heteroscedastic noise — variance grows with sqft to simulate real-world spread
    + np.random.normal(0, 0.02 + 0.00008 * sqft, n)
)

price = np.exp(log_price)

df = pd.DataFrame({
    "price":     price,
    "sqft":      sqft,
    "bedrooms":  bedrooms,
    "bathrooms": bathrooms,
    "age":       age,
    "dist_cbd":  dist_cbd,
    "garage":    garage,
})

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 ── OLS Regression via statsmodels
# ─────────────────────────────────────────────────────────────────────────────

# Design matrix: log-transform price for a semi-log hedonic specification
X = df[["sqft", "bedrooms", "bathrooms", "age", "dist_cbd", "garage"]]
X = sm.add_constant(X)           # adds the intercept column (const)
y = np.log(df["price"])          # log-price is the dependent variable

# Fit OLS — results object contains everything we need downstream
results = sm.OLS(y, X).fit()

print(results.summary())

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 ── Extract Residuals & Fitted Values from statsmodels
# ─────────────────────────────────────────────────────────────────────────────

# results.fittedvalues → array of ŷᵢ (predicted log-price for each observation)
# This is the in-sample prediction: ŷ = Xβ̂
fitted_values = results.fittedvalues          # pandas Series, index-aligned with df

# results.resid → array of eᵢ = yᵢ - ŷᵢ (raw OLS residuals)
# statsmodels computes this automatically; no manual subtraction needed
residuals = results.resid                     # pandas Series, same index

# Compute RMSE on the log-price scale
rmse = np.sqrt(mean_squared_error(y, fitted_values))
print(f"\n📐 RMSE (log-price scale): {rmse:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 ── Outlier Flagging (±2 Standard Deviations)
# ─────────────────────────────────────────────────────────────────────────────

# Standard deviation of the residual distribution
resid_std  = residuals.std()
resid_mean = residuals.mean()

# Boolean mask: True where |eᵢ| exceeds 2σ — these are the flagged outliers
outlier_mask = np.abs(residuals - resid_mean) > 2 * resid_std

# Create a categorical label column used by Plotly for colour mapping
outlier_label = np.where(outlier_mask, "Outlier (|e| > 2σ)", "Normal")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 ── Build the Forensics DataFrame
# ─────────────────────────────────────────────────────────────────────────────

forensics_df = pd.DataFrame({
    "Fitted (ŷ)":        fitted_values.values,
    "Residual (e)":      residuals.values,
    "Category":          outlier_label,
    "sqft":              df["sqft"].values,
    "bedrooms":          df["bedrooms"].values,
    "price_actual":      df["price"].values.round(0),
})

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 ── Scatter Plot: Fitted vs. Residuals (Main Diagnostic)
# ─────────────────────────────────────────────────────────────────────────────

# Color map: normal residuals → steel blue, outliers → crimson
color_map = {
    "Normal":             "#4A90D9",       # steel blue
    "Outlier (|e| > 2σ)": "#DC143C",       # crimson
}

fig_scatter = px.scatter(
    forensics_df,
    x="Fitted (ŷ)",
    y="Residual (e)",
    color="Category",
    color_discrete_map=color_map,
    hover_data={"sqft": True, "bedrooms": True, "price_actual": True},
    opacity=0.75,
    title="<b>Residual Forensics: Fitted Values vs. Residuals</b><br>"
          "<sup>Hedonic Pricing OLS — Log(Price) ~ Property Attributes</sup>",
    labels={
        "Fitted (ŷ)":   "Fitted Log-Price  ŷ",
        "Residual (e)": "Residual  eᵢ = yᵢ − ŷᵢ",
    },
    template="plotly_dark",
)

# ── Zero-line (the reference: where residuals should scatter around) ──────────
# A horizontal line at y=0 represents the OLS assumption E[e|X]=0.
# Systematic deviations above/below this line reveal model misspecification.
fig_scatter.add_hline(
    y=0,
    line_dash="dash",
    line_color="white",
    line_width=1.5,
    annotation_text="  Zero residual line",
    annotation_font_color="white",
    annotation_font_size=11,
)

# ── ±2σ bands (soft outlier thresholds) ──────────────────────────────────────
for sigma_val, label in [(2 * resid_std, "+2σ"), (-2 * resid_std, "−2σ")]:
    fig_scatter.add_hline(
        y=sigma_val,
        line_dash="dot",
        line_color="#FFA500",         # amber — visible but not dominant
        line_width=1,
        annotation_text=f"  {label}",
        annotation_font_color="#FFA500",
        annotation_font_size=10,
    )

fig_scatter.update_layout(
    font_family="Courier New",
    title_font_size=16,
    legend_title_text="Residual Category",
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#0f0f1a",
    xaxis=dict(gridcolor="#2a2a4a", zerolinecolor="#2a2a4a"),
    yaxis=dict(gridcolor="#2a2a4a", zerolinecolor="#2a2a4a"),
    margin=dict(t=100, b=60, l=70, r=40),
    height=550,
)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 ── Residual Distribution Histogram (Normality Check)
# ─────────────────────────────────────────────────────────────────────────────

fig_hist = px.histogram(
    forensics_df,
    x="Residual (e)",
    nbins=40,
    color="Category",
    color_discrete_map=color_map,
    title="<b>Residual Distribution</b><br><sup>Should approximate N(0, σ²) under OLS assumptions</sup>",
    template="plotly_dark",
    opacity=0.8,
    barmode="overlay",
)

fig_hist.add_vline(
    x=0,
    line_dash="dash",
    line_color="white",
    line_width=1.5,
    annotation_text="  μ = 0 (OLS guarantee)",
    annotation_font_color="white",
    annotation_font_size=10,
)

fig_hist.update_layout(
    font_family="Courier New",
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#0f0f1a",
    xaxis=dict(gridcolor="#2a2a4a"),
    yaxis=dict(gridcolor="#2a2a4a"),
    margin=dict(t=80, b=60, l=70, r=40),
    height=400,
)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 ── Scale-Location Plot (√|Residuals| vs Fitted)
#   Classic heteroscedasticity detector — if the spread fans out, you have it.
# ─────────────────────────────────────────────────────────────────────────────

forensics_df["√|Residual|"] = np.sqrt(np.abs(forensics_df["Residual (e)"]))

fig_scale = px.scatter(
    forensics_df,
    x="Fitted (ŷ)",
    y="√|Residual|",
    color="Category",
    color_discrete_map=color_map,
    trendline="lowess",               # LOWESS smoother reveals the heteroscedastic trend
    trendline_color_override="#FFD700",
    hover_data={"sqft": True},
    title="<b>Scale-Location Plot</b><br>"
          "<sup>√|eᵢ| vs ŷ — Fanning pattern = heteroscedasticity</sup>",
    template="plotly_dark",
    opacity=0.7,
)

fig_scale.update_layout(
    font_family="Courier New",
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#0f0f1a",
    xaxis=dict(gridcolor="#2a2a4a"),
    yaxis=dict(gridcolor="#2a2a4a"),
    margin=dict(t=80, b=60, l=70, r=40),
    height=450,
)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9 ── Render All Panels
# ─────────────────────────────────────────────────────────────────────────────

fig_scatter.show()
fig_hist.show()
fig_scale.show()

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10 ── Diagnostic Summary
# ─────────────────────────────────────────────────────────────────────────────

n_outliers   = outlier_mask.sum()
pct_outliers = 100 * n_outliers / n

print("\n" + "═" * 60)
print("  RESIDUAL FORENSICS — DIAGNOSTIC SUMMARY")
print("═" * 60)
print(f"  Observations      : {n}")
print(f"  Outliers (|e|>2σ) : {n_outliers}  ({pct_outliers:.1f}% of sample)")
print(f"  Residual Mean     : {resid_mean:.5f}  (should be ≈ 0)")
print(f"  Residual Std Dev  : {resid_std:.5f}")
print(f"  RMSE (log-scale)  : {rmse:.4f}")
print(f"  R²                : {results.rsquared:.4f}")
print(f"  Adj. R²           : {results.rsquared_adj:.4f}")
print(f"  AIC               : {results.aic:.2f}")
print(f"  BIC               : {results.bic:.2f}")
print("═" * 60)

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.784
Model:                            OLS   Adj. R-squared:                  0.780
Method:                 Least Squares   F-statistic:                     207.2
Date:                Tue, 10 Mar 2026   Prob (F-statistic):          7.99e-111
Time:                        19:03:33   Log-Likelihood:                 3.6002
No. Observations:                 350   AIC:                             6.800
Df Residuals:                     343   BIC:                             33.81
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         12.0207      0.066    181.459      0.0


════════════════════════════════════════════════════════════
  RESIDUAL FORENSICS — DIAGNOSTIC SUMMARY
════════════════════════════════════════════════════════════
  Observations      : 350
  Outliers (|e|>2σ) : 22  (6.3% of sample)
  Residual Mean     : 0.00000  (should be ≈ 0)
  Residual Std Dev  : 0.23984
  RMSE (log-scale)  : 0.2395
  R²                : 0.7838
  Adj. R²           : 0.7800
  AIC               : 6.80
  BIC               : 33.81
════════════════════════════════════════════════════════════
